# 05. RT-DETR (на запитах)

RT-DETR-L (Ultralytics, ваги COCO) порівнюємо з YOLO11s при однаковому бюджеті: ті самі кадри, роздільність 960 і кількість епох.
Щоб вкластися в ліміт GPU на Kaggle, для обох моделей беремо кожен 2-й кадр з `prepared/train` (це кожен 6-й кадр відео, ~4 тис. кадрів) і 20 епох.
RT-DETR вчиться за своїм рецептом (AdamW, lr 1e-4, weight decay 1e-4), YOLO за рецептом Ultralytics. Оцінка така сама, як в інших ноутбуках.

Запити декодера RT-DETR беруться як top-k ознак енкодера, тому кількість запитів можна міняти, не втрачаючи переднавчені ваги.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ostapmn/ucu_ai_hw_2.git"
if Path("/kaggle/working").exists() and not Path("common.py").exists():
    repo = Path("/kaggle/working/repo")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
    os.chdir(repo / "code")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, ".")
DEBUG = os.environ.get("HW2_DEBUG") == "1"

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch

import analysis as an
import common as c
import yolo_utils as yu
from ultralytics import RTDETR, YOLO

yu.setup(wandb=False)
c.set_seed()
P = c.paths()
OUT = P.out / "query_based"
print("відновлено з Input:", c.restore_from_inputs(P, "query_based"))
GT = P.prep / "val" / "annotations_eval.json"
RAW = P.data / "val" / "annotations_coco.json"
BUDGET = dict(epochs=20, imgsz=960, workers=4, deterministic=False, plots=True, device=yu.gpu_devices())
if DEBUG:
    BUDGET.update(epochs=1, imgsz=320, workers=0, plots=False, device="cpu")
DATA = yu.write_data_yaml(P.prep, OUT / "data_s6.yaml", train_every=20 if DEBUG else 2)
RTDETR_ARGS = dict(optimizer="AdamW", lr0=1e-4, weight_decay=1e-4, warmup_epochs=1, batch=4 if DEBUG else 8)

## 1. Експерименти

Медіана 43 об'єкти на кадр, максимум 180 (EDA), тому порівнюємо стандартні 300 запитів з 500. Окремо вимикаємо denoising-запити (DN-DETR), щоб побачити, як вони впливають на збіжність.

In [ ]:
EXPERIMENTS = {
    "yolo_s6": dict(rtdetr=False, model=None, weights="yolo11s.pt", args=dict(batch=4 if DEBUG else 16)),
    "rtdetr_q300": dict(rtdetr=True, model=yu.write_rtdetr_yaml(OUT / "rtdetr-l-q300.yaml", nq=300),
                        weights="rtdetr-l.pt", args=RTDETR_ARGS),
    "rtdetr_q500": dict(rtdetr=True, model=yu.write_rtdetr_yaml(OUT / "rtdetr-l-q500.yaml", nq=500),
                        weights="rtdetr-l.pt", args=RTDETR_ARGS),
    "rtdetr_q300_nodn": dict(rtdetr=True, model=yu.write_rtdetr_yaml(OUT / "rtdetr-l-q300-nodn.yaml", nq=300, nd=0),
                             weights="rtdetr-l.pt", args=RTDETR_ARGS),
}
RUN = ["yolo_s6", "rtdetr_q300", "rtdetr_q500"]


def run_experiment(name):
    cfg = EXPERIMENTS[name]
    args = {**BUDGET, **cfg["args"]}
    run = yu.train(cfg["model"], cfg["weights"], DATA, OUT, name, rtdetr=cfg["rtdetr"], **args)
    model = (RTDETR if cfg["rtdetr"] else YOLO)(run / "weights" / "best.pt")
    dets, _ = yu.predict_prepared(model, P.prep, "val", imgsz=args["imgsz"], batch=8)
    c.save_json(dets, OUT / name / "val_dets.json")
    metrics = c.evaluate(GT, dets, per_class=True)
    ms = yu.measure_speed(model, P.prep, imgsz=args["imgsz"], n=5 if DEBUG else 100)
    hours = float(yu.results_csv(run)["time"].iloc[-1]) / 3600
    ngpu = len(args["device"]) if isinstance(args["device"], list) else 1
    family = "query-based" if cfg["rtdetr"] else "budget-matched CNN"
    return an.log_result(P.results, name, family, metrics, ms_per_img=ms, epochs=args["epochs"],
                         wall_hours=hours, gpu_hours=hours * ngpu, imgsz=args["imgsz"], config=str(cfg["args"]))


for name in RUN:
    print("=" * 20, name)
    run_experiment(name)
    torch.cuda.empty_cache()

## 2. Збіжність при однаковому бюджеті

Криві з `results.csv` Ultralytics: val mAP50-95 після кожної епохи. Ця валідація не знає про ignore-регіони, тому цифри нижчі, ніж у `common.evaluate`, але моделі між собою порівнювати можна. Праворуч та сама крива за часом навчання.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.4))
for name in RUN:
    if not (OUT / name / "results.csv").exists():
        continue
    r = yu.results_csv(OUT / name)
    axes[0].plot(r.epoch, r["metrics/mAP50-95(B)"], marker=".", label=name)
    axes[1].plot(r["time"] / 3600, r["metrics/mAP50-95(B)"], marker=".", label=name)
    loss = [col for col in r.columns if col.startswith("train/") and ("giou" in col or "box" in col)][0]
    axes[2].plot(r.epoch, r[loss], label=f"{name}: {loss}")
axes[0].set_xlabel("епоха")
axes[1].set_xlabel("години навчання (wall clock)")
axes[2].set_xlabel("епоха")
for ax, t in zip(axes, ["val mAP50-95", "val mAP50-95 від часу", "box/GIoU loss (train)"]):
    ax.set_title(t)
axes[0].legend()
axes[2].legend(fontsize=7)
fig.savefig(P.images / "rtdetr_vs_cnn_curves.png", bbox_inches="tight", dpi=130)
plt.show()

## 3. Результати, дрібні об'єкти і щільні сцени

In [ ]:
COLS = ["family", "mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR_max", "ms_per_img", "gpu_hours"]
res = an.all_results(P).set_index("name")
mine = res.loc[[n for n in RUN if n in res.index]]
display(mine[COLS].round(3))
display(mine[[f"AP_{k}" for k in c.CLASSES]].round(3))

In [ ]:
dens = {}
for name in RUN:
    path = OUT / name / "val_dets.json"
    if path.exists():
        dens[name] = an.map_by_density(GT, c.load_json(path))["mAP"]
dens = pd.DataFrame(dens).rename_axis("об'єктів на кадрі")
c.save_json(dens.reset_index().astype(str).to_dict("records"), OUT / "map_by_density.json")
dens.round(3)

## 4. Помилки найкращого RT-DETR

In [ ]:
best = mine[mine.family == "query-based"].mAP.idxmax()
dets = c.load_json(OUT / best / "val_dets.json")
d_df, g_df = an.error_report(GT, dets, P.data / "val", RAW, P.images, prefix="rtdetr", score_thr=0.3)

## Висновки

Усі числа з таблиць цього ноутбука (val, `common.evaluate`), по одному запуску на конфігурацію. Бюджет однаковий: 20 епох, кожен 2-й кадр з `prepared/train`, 960 px. Ultralytics-валідація в логах не бачить ignore-регіонів, тому її цифри нижчі.

**Збіжність (розділ 2).**
- yolo_s6: mAP 0.178, AP50 0.371, AP_small 0.102, 0.9 GPU-год, 18 мс на кадр.
- rtdetr_q500: 0.175, 0.344, 0.088, AR_max 0.416 (найвищий), 3.6 GPU-год, 128 мс.
- rtdetr_q300: 0.068, 0.147, 0.028, 3.4 GPU-год.

`q500` дійшов до рівня YOLO за mAP, але навчання коштувало вчетверо більше GPU-годин, а інференс у 7 разів повільніший (FP32, batch 1). `q300` не зійшовся: перші 4 епохи Ultralytics-валідація показувала нуль (`cls_loss` ≈ 2e-5, клас майже не вчився), далі mAP росте повільно і на 20-й епосі досягає лише 0.065 (mAP50-95), хоча крива ще йшла вгору. `q500` вчився з першої епохи і закінчив із 0.166. Причину різниці з кількості запитів пояснити не можна: запуск один, а схоже радше на повільний старт навчання `q300`. Варіант без denoising (`rtdetr_q300_nodn`) описаний у коді, але в запуск не входив, тому його вплив не перевірено.

**Bipartite matching і NMS.** RT-DETR не потребує NMS, але при score ≥ 0.3 у його детекціях 15.1% дублікатів (YOLO tal13 3.3%, Faster R-CNN 1.4%, RetinaNet 1.7%). Нашого бюджету вистачило, щоб `q500` зійшовся за mAP, але не щоб matching повністю прибрав дублікати.

**Малі об'єкти і щільні сцени (розділ 3).** AP_small 0.088 проти 0.102 у YOLO, AP_medium 0.158 проти 0.166, AP_large однаковий (0.309 і 0.313). mAP за кількістю об'єктів на кадрі (YOLO / q500): (0, 20] 0.055 / 0.115, (20, 40] 0.237 / 0.225, (40, 80] 0.210 / 0.186, (80, 1000] 0.231 / 0.230. 500 запитів не обмежують щільні сцени, а на розріджених кадрах RT-DETR кращий удвічі. `q300` у всіх групах близько 0.05 до 0.08. За класами `q500` кращий на van (0.202 проти 0.164) і bus (0.088 проти 0.040), гірший на person, bicycle і motor.

**Помилки `q500` (score ≥ 0.3):** tp 34.2%, cls 13.1%, loc 6.3%, bg 16.5%, ignore 10.9%, dup 15.1%. Recall: малі 0.45, середні 0.65, великі 0.79; bicycle 0.53 і motor 0.53 вищі, ніж у Faster R-CNN (0.13 і 0.38).

**Висновок.** При бюджеті 20 епох RT-DETR-L (500 запитів) не перевершує YOLO11s ні за mAP, ні за швидкістю, ні за вартістю навчання. Його перевага: найвищий AR_max і краща робота на van, bus та розріджених кадрах. Тому для фінального pipeline YOLO дешевший, а RT-DETR може стати кандидатом для ансамблю.